# BirdCLEF 2026 — Inference v21 (Perch-only)
## PerchHead (v21, domain-fixed) ONLY

**This version disables all mel model logic and uses only PerchHead for inference.**

- Based on v21 ensemble notebook
- For diagnostic use: isolates PerchHead contribution


# BirdCLEF 2026 — Inference v21
## ResNet18 (v20) + EfficientNet-B0 (v20) + PerchHead (v21, domain-fixed)
### 15 models total: 3 archs × 5 folds

**What changed vs v20:**
- `perch_head` weights retrained on in-domain soundscape embeddings (v21)
- Fixes train/inference domain mismatch that caused v20 regression

**Required Kaggle input datasets:**
- `birdclef-2026` (competition data)
- `chiragggg/birdclef-2026-v20-model` (mel model weights: resnet18_v20 + efficientnet_b0_v20)
- `chiragggg/birdclef-2026-perch-weights-v21` (perch_head_v21_fold*.pt)
- `google/bird-vocalization-classifier` (Perch 2.0 TF model, TF2 > perch_v2 > v2)


**Optional (recommended for submission — skips TF embedding step entirely):**
- `chiragggg/birdclef-2026-test-embs-v21` (pre-computed Perch embeddings from GCP precompute notebook)

In [ ]:
# === CELL 0: ENVIRONMENT CHECK ===
# NOTE: pip install does NOT work in Kaggle submission notebooks (no internet).
# To use Perch embeddings in submission, run birdclef2026-perch-precompute-gcp-v2.ipynb
# on GCP/locally and upload the .npy output as dataset 'birdclef-2026-test-embs-v21'.
import sys, subprocess
result = subprocess.run([sys.executable, "-c", "import tensorflow; print(tensorflow.__version__)"], capture_output=True, text=True)
print(f"TensorFlow version: {result.stdout.strip() or 'not found'}")

In [1]:
# === CELL 1: IMPORTS & CONFIG ===
import os, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

# Hide GPU from TensorFlow (Perch runs CPU-only at inference)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
try:
    import tensorflow as tf
    _TF_OK = True
except Exception:
    _TF_OK = False

import torch
import torch.nn as nn
import timm
from torch.cuda.amp import autocast
from tqdm import tqdm
import gc

warnings.filterwarnings("ignore")

CFG = dict(
    sr            = 16000,
    seconds       = 5,
    n_mels        = 64,
    n_fft         = 1024,
    hop           = 320,
    fmin          = 60,
    fmax          = 8000,
    mel_archs     = ["resnet18", "efficientnet_b0"],   # v20 weights
    folds         = 5,
    tta_offsets   = [-1.25, 0.0, 1.25],
    device        = "cuda" if torch.cuda.is_available() else "cpu",
    perch_sr      = 32000,
    perch_emb_dim = 1536,
)

device = torch.device(CFG["device"])
torch.set_num_threads(os.cpu_count() or 4)
n_mel_models   = len(CFG["mel_archs"]) * CFG["folds"]
n_perch_models = CFG["folds"]
print("v21 inference config ready")
print(f"   Device          : {device}")
print(f"   TF OK           : {_TF_OK}")
print(f"   Mel models (v20): {n_mel_models}  ({CFG['mel_archs']} x {CFG['folds']} folds)")
print(f"   Perch models    : {n_perch_models}  (perch_head_v21 x {CFG['folds']} folds)")
print(f"   TTA crops       : {len(CFG['tta_offsets'])}")


E0000 00:00:1778723858.414578      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778723858.524228      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778723859.494745      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778723859.494827      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778723859.494832      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778723859.494836      16 computation_placer.cc:177] computation placer already registered. Please check linka

v21 inference config ready
   Device          : cpu
   TF OK           : True
   Mel models (v20): 10  (['resnet18', 'efficientnet_b0'] x 5 folds)
   Perch models    : 5  (perch_head_v21 x 5 folds)
   TTA crops       : 3


In [ ]:
# === CELL 2: DATA PATHS & SPECIES ===
def _first_existing(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TAXONOMY_CSV = _first_existing(
    "/kaggle/input/birdclef-2026/taxonomy.csv",
    "/kaggle/input/competitions/birdclef-2026/taxonomy.csv",
)
TEST_AUDIO = _first_existing(
    "/kaggle/input/birdclef-2026/test_soundscapes",
    "/kaggle/input/competitions/birdclef-2026/test_soundscapes",
)
SAMPLE_SUB = _first_existing(
    "/kaggle/input/birdclef-2026/sample_submission.csv",
    "/kaggle/input/competitions/birdclef-2026/sample_submission.csv",
)
MEL_CKPT_DIR = _first_existing(
    "/kaggle/input/birdclef-2026-v20-model",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-v20-model",
    "/kaggle/working",
)
PERCH_CKPT_DIR = _first_existing(
    "/kaggle/input/birdclef-2026-perch-weights-v21",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v21",
    "/kaggle/working",
)
PERCH_MODEL_PATH = Path(_first_existing(
    # perch_v2/1 must be tried BEFORE v2 — Kaggle TF build only supports XlaCallModule <= v9
    # perch_v2/2 was compiled with v10 and causes "XlaCallModuleOp version 10 not supported"
    "/kaggle/input/bird-vocalization-classifier/tensorflow2/perch_v2/1",
    "/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/1",
    "/kaggle/input/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/1",
    "/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/1",
    # v2 as last resort (will fail on current Kaggle TF build)
    "/kaggle/input/bird-vocalization-classifier/tensorflow2/perch_v2/2",
    "/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2",
    "/kaggle/input/bird-vocalization-classifier",
))

# Pre-computed embeddings — try both dataset slug variants
_PRECOMP_CANDIDATES = [
    "/kaggle/input/birdclef-2026-perch-embs-v2",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-perch-embs-v2",
    "/kaggle/input/birdclef-2026-test-embs-v21",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-test-embs-v21",
]
_precomp = next((p for p in _PRECOMP_CANDIDATES if os.path.exists(p) and list(Path(p).glob("*.npy"))), None)
if _precomp:
    EMBD_DIR = Path(_precomp)
    print(f"✅ Using pre-computed embeddings: {EMBD_DIR}  ({len(list(EMBD_DIR.glob('*.npy')))} files)")
else:
    EMBD_DIR = Path("/kaggle/working/test_embs")
    EMBD_DIR.mkdir(parents=True, exist_ok=True)
    print(f"   EMBD_DIR (runtime): {EMBD_DIR}")

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df["primary_label"].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)
print(f"   TAXONOMY_CSV    : {TAXONOMY_CSV}")
print(f"   TEST_AUDIO      : {TEST_AUDIO}")
print(f"   MEL_CKPT_DIR    : {MEL_CKPT_DIR}")
print(f"   PERCH_CKPT_DIR  : {PERCH_CKPT_DIR}")
print(f"✅ {n_classes} species loaded")


✅ 234 species from taxonomy.csv
   TEST_AUDIO      : /kaggle/input/competitions/birdclef-2026/test_soundscapes
   MEL_CKPT_DIR    : /kaggle/input/datasets/chiragggg/birdclef-2026-v20-model
   PERCH_CKPT_DIR  : /kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v21
   PERCH_MODEL_PATH: /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2  (exists=True)


In [3]:
# === CELL 3: MEL HELPER ===
def logmel_from_wave(wave: np.ndarray, sr: int) -> np.ndarray:
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr,
        n_fft=CFG["n_fft"],
        hop_length=CFG["hop"],
        n_mels=CFG["n_mels"],
        fmin=CFG["fmin"],
        fmax=CFG["fmax"],
        power=2.0,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min, S_max = S_db.min(), S_db.max()
    if S_max - S_min < 1e-9:
        return np.zeros_like(S_db, dtype=np.float32)
    return np.clip((S_db - S_min) / (S_max - S_min + 1e-9), 0.0, 1.0).astype(np.float32)

print("✅ logmel_from_wave defined")


✅ logmel_from_wave defined


In [4]:
# === CELL 4: MODEL ARCHITECTURES ===
class BirdCLEFModel(nn.Module):
    def __init__(self, arch: str, n_classes: int, pretrained: bool = False):
        super().__init__()
        self.arch = arch
        if arch == "resnet18":
            self.backbone = timm.create_model("resnet18", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features
        elif arch == "resnet50":
            self.backbone = timm.create_model("resnet50", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features
        elif arch in ("efficientnet_b0", "efficientnet_b2"):
            self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0)
            stem = self.backbone.conv_stem
            self.backbone.conv_stem = nn.Conv2d(
                1, stem.out_channels,
                kernel_size=stem.kernel_size, stride=stem.stride,
                padding=stem.padding, bias=False,
            )
            in_features = self.backbone.num_features
        else:
            raise ValueError(f"Unsupported arch: {arch}")

        # v20 head: Linear(in→512) → ReLU → Dropout(0.4) → Linear(512→n_classes)
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))


class PerchHead(nn.Module):
    """MLP head that runs on top of Perch 2.0 1536-dim embeddings."""
    def __init__(self, n_classes: int, emb_dim: int = 1536):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

print("✅ BirdCLEFModel and PerchHead defined")


✅ BirdCLEFModel and PerchHead defined


In [5]:
# === CELL 5: LOAD CHECKPOINTS (Perch-only) ===
mel_models  = []   # Perch-only: mel models disabled
emb_models  = []   # perch_head_v21
missing     = []

# Perch-only: skip loading mel models
# Load perch_head models (v21 weights)
for fold_idx in range(CFG["folds"]):
    ckpt_path = Path(PERCH_CKPT_DIR) / f"perch_head_v21_fold{fold_idx}.pt"
    if not ckpt_path.exists():
        missing.append(str(ckpt_path))
        continue
    m = PerchHead(n_classes=n_classes, emb_dim=CFG["perch_emb_dim"]).to(device)
    m.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False))
    m.eval()
    emb_models.append(m)
    print(f"   ✅ Loaded {ckpt_path.name}")

if missing:
    print(f"\n⚠️  {len(missing)} checkpoint(s) not found:")
    for p in missing:
        print(f"      {p}")

total_models = len(mel_models) + len(emb_models)
print(f"\n✅ {len(mel_models)} mel models + {len(emb_models)} perch models = {total_models} total")


   ✅ Loaded perch_head_v21_fold0.pt
   ✅ Loaded perch_head_v21_fold1.pt
   ✅ Loaded perch_head_v21_fold2.pt
   ✅ Loaded perch_head_v21_fold3.pt
   ✅ Loaded perch_head_v21_fold4.pt

✅ 0 mel models + 5 perch models = 5 total


In [ ]:
# === CELL 6: LOAD PERCH MODEL (or use pre-computed test embeddings) ===
# Kaggle scoring: test_soundscapes/ is EMPTY at commit time — only populated during scoring.
# Therefore we NEVER precompute all embeddings upfront in this cell.
# Strategy:
#   1. If a pre-computed test embedding dataset is attached AND covers test IDs -> use it (fast path).
#   2. Otherwise load the Perch TF model into memory now; compute embeddings
#      on-the-fly per soundscape inside predict_soundscape() during scoring.

_PERCH_BATCH_INF  = 16
_perch_ready_inf  = False
_perch_infer      = None   # TF inference fn — kept alive for on-the-fly computation
_perch_emb_key    = 'embedding'  # updated after model load
_perch_target     = CFG['perch_sr'] * CFG['seconds']  # 160,000 samples

# --- Path 1: pre-computed test embeddings ---
_existing_embs = len(list(EMBD_DIR.glob('*.npy')))
if _existing_embs > 0:
    _sample_sub_check = pd.read_csv(SAMPLE_SUB)
    _test_sc_ids = set(_sample_sub_check['row_id'].str.rsplit('_', n=1).str[0])
    _emb_stems   = {f.stem for f in EMBD_DIR.glob('*.npy')}
    _conv_b_ids  = {s.rsplit('_', 1)[0] for s in _emb_stems if s.rsplit('_', 1)[-1].isdigit()}
    _conv_a_ids  = {s.rsplit('.', 1)[0] if '.' in s else s for s in _emb_stems}
    _covered     = _test_sc_ids & (_conv_b_ids | _conv_a_ids)
    if _covered:
        print(f'\u2705 Pre-computed test embeddings: {_existing_embs} files, {len(_covered)}/{len(_test_sc_ids)} soundscapes covered')
        _perch_ready_inf = True
    else:
        print(f'\u26a0\ufe0f  {_existing_embs} .npy files in EMBD_DIR but none match test IDs.')
        print('   Likely TRAIN embeddings attached by mistake. Will fall through to TF on-the-fly.')

# --- Path 2: load TF model for on-the-fly computation ---
if not _perch_ready_inf:
    if not _TF_OK:
        print('\u26a0\ufe0f  TensorFlow not available \u2014 perch_head will output neutral 0.5')
    elif not PERCH_MODEL_PATH.exists():
        print(f'\u26a0\ufe0f  Perch TF model not found at {PERCH_MODEL_PATH}')
        print("   Add 'google/bird-vocalization-classifier' (TF2 > perch_v2 > v2) as a notebook input.")
    else:
        try:
            print(f'Loading Perch 2.0 from {PERCH_MODEL_PATH} ...')
            tf.config.set_visible_devices([], 'GPU')
            _perch_model = tf.saved_model.load(str(PERCH_MODEL_PATH))
            _perch_infer = _perch_model.signatures['serving_default']
            # Key is always "embedding" for all known Perch variants.
            # Do NOT run a dummy forward pass here — it triggers XLA compilation
            # which fails with "XlaCallModuleOp version 10 not supported" on perch_v2/2.
            _perch_emb_key = 'embedding'
            print(f'\u2705 Perch TF model loaded. Emb key: "{_perch_emb_key}" (no dummy pass)')
            print('   Embeddings will be computed on-the-fly per soundscape during scoring.')
            _perch_ready_inf = True
        except Exception as _e:
            import traceback; traceback.print_exc()
            print(f'\u26a0\ufe0f  Failed to load Perch model: {_e}')

if not _perch_ready_inf:
    print('\u26a0\ufe0f  No Perch available \u2014 predictions will be neutral 0.5')


Loading Perch 2.0 from /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2 ...
  Perch output keys: ['embedding', 'spatial_embedding', 'spectrogram', 'label']
  Sample soundscape IDs: ['BC2026_Test_0001_S05_20250227_010002']
  TEST_AUDIO dir       : /kaggle/input/competitions/birdclef-2026/test_soundscapes
  Sample paths checked : ['/kaggle/input/competitions/birdclef-2026/test_soundscapes/BC2026_Test_0001_S05_20250227_010002.ogg']


Perch embed (test): 100%|██████████| 1/1 [00:00<00:00, 113.65it/s]

  Files in TEST_AUDIO: ['readme.txt']


Test Perch embeddings: 0 saved  (1 soundscapes failed)
TF model freed — proceeding to PyTorch inference
perch_head will output neutral 0.5


Traceback (most recent call last):
  File "/tmp/ipykernel_16/2680961880.py", line 132, in <cell line: 0>
    raise RuntimeError("Perch embeddings were NOT computed! Submission will not use Perch.")
RuntimeError: Perch embeddings were NOT computed! Submission will not use Perch.


In [ ]:
# === CELL 7: PREDICT FUNCTION (Perch on-the-fly or pre-computed cache) ===
_use_amp = (device.type == 'cuda')

# Build lookup index from pre-computed embedding files (if any attached).
# Convention B: {sc_id}_{end_s}.npy  (runtime-computed by runtime precompute)
# Convention A: {sc_id}.ogg.npy      (per-soundscape focal-style)
_emb_index: dict = {}
if EMBD_DIR.exists():
    for _f in EMBD_DIR.glob('*.npy'):
        _stem  = _f.stem
        _parts = _stem.rsplit('_', 1)
        if len(_parts) == 2 and _parts[1].isdigit():
            _emb_index[(_parts[0], int(_parts[1]))] = _f   # Convention B
        else:
            _sid = _stem.rsplit('.', 1)[0] if '.' in _stem else _stem
            _emb_index[_sid] = _f                          # Convention A

print(f'\u2705 Embedding index: {len(_emb_index)} pre-computed entries')
if _emb_index:
    _sk = next(iter(_emb_index))
    print(f'   Example key: {_sk!r}  ({"per-window" if isinstance(_sk, tuple) else "per-soundscape"})')
else:
    print('   (will compute all embeddings on-the-fly from audio during scoring)')


def _compute_embs_from_audio(audio_path: str, end_secs_list: list) -> dict:
    """Compute Perch embeddings for a list of end_seconds from one audio file.
    Returns {end_s: np.ndarray(1536,)} for successfully computed windows.
    Requires _perch_infer to be loaded (Cell 6 Path 2).
    """
    result = {}
    if _perch_infer is None:
        return result
    try:
        y, sr0 = sf.read(audio_path, always_2d=False)
        if y.ndim == 2:
            y = y.mean(axis=1)
        if sr0 != CFG['perch_sr']:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr0, target_sr=CFG['perch_sr'])
        y = y.astype(np.float32)
        clips = []
        for es in end_secs_list:
            end_samp   = int(es * CFG['perch_sr'])
            start_samp = max(0, end_samp - _perch_target)
            clip = y[start_samp:end_samp]
            if len(clip) < _perch_target:
                clip = np.pad(clip, (0, _perch_target - len(clip)))
            clips.append(clip)
        all_embs = []
        for i in range(0, len(clips), _PERCH_BATCH_INF):
            batch = np.stack(clips[i:i + _PERCH_BATCH_INF])
            out   = _perch_infer(inputs=tf.constant(batch, dtype=tf.float32))
            embs  = out[_perch_emb_key].numpy()
            if embs.ndim == 3:
                embs = embs.mean(axis=1)
            all_embs.append(embs)
        all_embs = np.vstack(all_embs)
        for es, emb in zip(end_secs_list, all_embs):
            result[es] = emb.astype(np.float32)
    except Exception as _ce:
        pass  # Caller will fall back to zeros for missing entries
    return result


def predict_soundscape(audio_path: str, end_seconds, soundscape_id: str = None) -> np.ndarray:
    end_seconds = list(end_seconds)
    n_rows      = len(end_seconds)
    neutral     = np.full((n_rows, n_classes), 0.5, dtype=np.float32)

    if not emb_models or not _perch_ready_inf:
        return neutral

    # 1. Check pre-computed cache for each window
    embs_for_windows: dict = {}
    missing_es = []
    for es in end_seconds:
        found = None
        for key in [(soundscape_id, es), soundscape_id]:
            if key in _emb_index:
                found = np.load(str(_emb_index[key]))
                break
        if found is not None:
            embs_for_windows[es] = found
        else:
            missing_es.append(es)

    # 2. Compute on-the-fly for windows not in cache
    if missing_es and audio_path is not None:
        computed = _compute_embs_from_audio(audio_path, missing_es)
        embs_for_windows.update(computed)

    # 3. Build embedding batch in original order
    emb_list = [
        embs_for_windows.get(es, np.zeros(CFG['perch_emb_dim'], dtype=np.float32))
        for es in end_seconds
    ]
    emb_batch = torch.from_numpy(np.stack(emb_list)).float().to(device)

    # 4. Run PerchHead ensemble
    all_model_probs = []
    for m in emb_models:
        with torch.inference_mode(), autocast(enabled=_use_amp):
            p = torch.sigmoid(m(emb_batch)).float().cpu().numpy()
        all_model_probs.append(p)

    return np.mean(all_model_probs, axis=0).astype(np.float32) if all_model_probs else neutral


print('\u2705 predict_soundscape() defined (on-the-fly Perch + pre-computed cache fallback)')
print(f'   Perch models : {len(emb_models)}  |  perch_ready={_perch_ready_inf}')
print(f'   TF on-the-fly: {_perch_infer is not None}')


predict_soundscape() defined (Perch-only)
   Perch models    : 5  (perch_ready=False)


In [8]:
# === CELL 8: GENERATE PREDICTIONS ===
sample_sub = pd.read_csv(SAMPLE_SUB)
print(f"Sample submission rows: {len(sample_sub)}")

all_row_ids    = []
all_probs_list = []
missing_audio  = 0

sample_sub = sample_sub.copy()
sample_sub["_soundscape_id"] = sample_sub["row_id"].str.rsplit("_", n=1).str[0]

for soundscape_id, group in tqdm(
    sample_sub.groupby("_soundscape_id"),
    desc="Soundscapes",
    unit="file",
):
    audio_path = None
    for ext in [".ogg", ".wav", ".flac"]:
        cand = Path(TEST_AUDIO) / f"{soundscape_id}{ext}"
        if cand.exists():
            audio_path = str(cand)
            break

    row_ids = [str(r) for r in group["row_id"]]

    if audio_path is None:
        missing_audio += 1
        all_row_ids.extend(row_ids)
        all_probs_list.append(np.full((len(row_ids), n_classes), 0.5, dtype=np.float32))
        continue

    try:
        end_seconds = [int(rid.rsplit("_", 1)[-1]) for rid in row_ids]
    except Exception as _e:
        print(f"WARNING: could not parse end_seconds: {_e}")
        missing_audio += 1
        all_row_ids.extend(row_ids)
        all_probs_list.append(np.full((len(row_ids), n_classes), 0.5, dtype=np.float32))
        continue

    try:
        probs_all = predict_soundscape(audio_path, end_seconds, soundscape_id=soundscape_id)
    except Exception as _e:
        print(f"WARNING: predict_soundscape failed for {soundscape_id}: {_e}")
        probs_all = np.full((len(row_ids), n_classes), 0.5, dtype=np.float32)

    all_row_ids.extend(row_ids)
    all_probs_list.append(probs_all)

if missing_audio:
    print(f"⚠️  {missing_audio} soundscape(s) not found — used neutral 0.5")
print(f"\n✅ Generated {len(all_row_ids)} prediction rows")


Sample submission rows: 3


Soundscapes: 100%|██████████| 1/1 [00:00<00:00, 196.11file/s]

⚠️  1 soundscape(s) not found — used neutral 0.5

✅ Generated 3 prediction rows


In [9]:
# === CELL 9: BUILD & SAVE SUBMISSION ===
probs_matrix = np.concatenate(all_probs_list, axis=0)   # (N_rows, n_classes)

sub_df = pd.DataFrame(probs_matrix, columns=species)
sub_df.insert(0, "row_id", all_row_ids)

# Align to sample submission column order
sample_cols = pd.read_csv(SAMPLE_SUB, nrows=0).columns.tolist()
sub_df = sub_df[sample_cols]

out_path = "/kaggle/working/submission.csv"
sub_df.to_csv(out_path, index=False)
print(f"✅ Submission saved: {out_path}")
print(f"   Shape : {sub_df.shape}")
print(sub_df.head(3))


✅ Submission saved: /kaggle/working/submission.csv
   Shape : (3, 235)
                                    row_id  1161364  116570  1176823  1491113  \
0   BC2026_Test_0001_S05_20250227_010002_5      0.5     0.5      0.5      0.5   
1  BC2026_Test_0001_S05_20250227_010002_10      0.5     0.5      0.5      0.5   
2  BC2026_Test_0001_S05_20250227_010002_15      0.5     0.5      0.5      0.5   

   1595929  209233  22930  22956  22961  ...  whnjay1  whtdov  whwpic1  \
0      0.5     0.5    0.5    0.5    0.5  ...      0.5     0.5      0.5   
1      0.5     0.5    0.5    0.5    0.5  ...      0.5     0.5      0.5   
2      0.5     0.5    0.5    0.5    0.5  ...      0.5     0.5      0.5   

   y00678  yebcar  yebela1  yecmac  yecpar  yehcar1  yeofly1  
0     0.5     0.5      0.5     0.5     0.5      0.5      0.5  
1     0.5     0.5      0.5     0.5     0.5      0.5      0.5  
2     0.5     0.5      0.5     0.5     0.5      0.5      0.5  

[3 rows x 235 columns]


In [ ]:
# === DEBUG CELL: Inspect Perch Embedding and Model Output ===
import random

# List available embedding files
emb_files = list(EMBD_DIR.glob('*.npy'))
print(f"Found {len(emb_files)} embedding files. Example: {emb_files[:3]}")

if emb_files:
    # Pick a random embedding file
    emb_path = random.choice(emb_files)
    emb = np.load(str(emb_path))
    print(f"Sample embedding shape: {emb.shape}, mean: {emb.mean():.4f}, std: {emb.std():.4f}")
    print(f"Embedding file: {emb_path.name}")

    # Run through PerchHead model
    emb_tensor = torch.from_numpy(emb).float().unsqueeze(0).to(device)
    for i, m in enumerate(emb_models):
        with torch.inference_mode():
            out = torch.sigmoid(m(emb_tensor)).cpu().numpy()
        print(f"Model {i} output (min={out.min():.4f}, max={out.max():.4f}, mean={out.mean():.4f})")
else:
    print("No embedding files found in EMBD_DIR. Check embedding generation step.")
